# Nettoyage des données

## Import de la base de données




In [ ]:
from pathlib import Path

import pandas as pd

def trouver_racine_projet():
    courant = Path.cwd().resolve()
    for dossier in (courant, *courant.parents):
        if (dossier / 'Data brutes').is_dir():
            return dossier
    raise FileNotFoundError(
        'Dossier « Data brutes » introuvable. Lancez le notebook depuis le projet.'
    )


RACINE = trouver_racine_projet()
DOSSIER_BRUT = RACINE / 'Data brutes'
DOSSIER_SORTIE = RACINE / 'Data nettoyees'
CHEMIN_PRIVE = DOSSIER_BRUT / 'DS_DERA_PRIVE_SERIES_LONGUES_data.csv'

if not CHEMIN_PRIVE.is_file():
    raise FileNotFoundError(f'Fichier privé brut manquant : {CHEMIN_PRIVE}')

print(f'Projet : {RACINE}')
print(f'Pandas : {pd.__version__}')


## 2. Renomage des variables



In [2]:
RENOMMAGE = {
    'GEO': 'geo',
    'GEO_OBJECT': 'zone_geo',
    'FREQ': 'frequence',
    'ACTIVITY': 'secteur_activite',
    'AGE': 'tranche_age',
    'NUMBER_EMPL': 'taille_entreprise',
    'PCS_ESE': 'categorie_socio_prof',
    'SEX': 'sexe',
    'WKTIME': 'temps_travail',
    'QUANTILE': 'centile_salaire',
    'DERA_MEASURE': 'indicateur',
    'UNIT_MEASURE': 'unite',
    'TIME_PERIOD': 'annee',
    'OBS_VALUE': 'valeur',
}

MODALITES = {
    'sexe': {'F': 'Femme', 'M': 'Homme', '_T': 'Ensemble'},
    'categorie_socio_prof': {
        '3': 'Cadres et prof. intellectuelles sup.',
        '4': 'Professions intermédiaires',
        '5': 'Employés',
        '6': 'Ouvriers',
        '_T': 'Ensemble',
    },
    'tranche_age': {
        'Y_LT26': 'Moins de 26 ans',
        'Y26T30': 'De 26 à 30 ans',
        'Y31T40': 'De 31 à 40 ans',
        'Y41T50': 'De 41 à 50 ans',
        'Y51T60': 'De 51 à 60 ans',
        'Y_GT60': 'Plus de 60 ans',
        '_T': 'Ensemble',
    },
    'taille_entreprise': {
        'E_LT10': 'Moins de 10 salariés',
        'E10T49': '10 à 49 salariés',
        'E50T199': '50 à 199 salariés',
        'E200T499': '200 à 499 salariés',
        'E500T999': '500 à 999 salariés',
        'E_GE1000': '1000 salariés et plus',
        '_T': 'Ensemble',
    },
    'unite': {
        'XDC': 'Euros constants',
        'PT': 'Pourcentage',
        'IX': 'Indice base 100',
    },
    'temps_travail': {'FT': 'Temps complet', '_T': 'Ensemble'},
    'secteur_activite': {
        'H': 'Transports et entreposage',
        'I': 'Hébergement et restauration',
        'J': 'Information et communication',
        'K': 'Activités financières et d’assurance',
        'L': 'Activités immobilières',
        'M_N': 'Activités spécialisées, scientifiques, techniques et services support (M_N)',
        'OTQ': 'Administration publique, enseignement, santé et action sociale (OTQ)',
        'R_S': 'Arts, divertissement, loisirs et autres services (R_S)',
        'B_D_E': 'Industries extractives, énergie, eau, déchets (B_D_E)',
        'C10T12': 'Fabrication d’aliments, boissons et tabac (C10T12)',
        'C19': 'Cokéfaction et raffinage (C19)',
        'C26T28': 'Fabrication de produits informatiques, électroniques et machines (C26T28)',
        'C29_30': 'Fabrication de matériels de transport (C29_30)',
        'C_OTH': 'Autres industries manufacturières (C_OTH)',
        'F': 'Construction',
        'G': 'Commerce ; réparation d’automobiles et de motocycles',
        '_T': 'Ensemble des secteurs',
    },
    'indicateur': {
        'SALAIRE_NET_EQTP_MENSUEL_MOYEN_EUROS_CONSTANTS': 'Salaire net moyen mensuel (Euros constants)',
        'SALAIRE_NET_EQTP_MENSUEL_MOYEN_EVOLUTION_CONSTANT': 'Évolution du salaire net mensuel (%)',
        'SALAIRE_NET_EQTP_MENSUEL_MOYEN_INDICE_100_1996': 'Indice du salaire net mensuel (Base 100 en 1996)',
        'PARTMS_10': 'Part de la masse salariale perçue par le top 10%',
        'PARTMS_100': 'Part de la masse salariale perçue par le top 1%',
        'PARTMS_1000': 'Part de la masse salariale perçue par le top 0,1%',
        'PARTMS_10000': 'Part de la masse salariale perçue par le top 0,01%',
    },
    'centile_salaire': {
        'CENTILE_10': '10e centile (D1 - 10% gagnent moins)',
        'CENTILE_25': '1er quartile (Q1 - 25% gagnent moins)',
        'CENTILE_50': 'Médiane (50% gagnent moins/plus)',
        'CENTILE_75': '3e quartile (Q3 - 75% gagnent moins)',
        'CENTILE_90': '90e centile (D9 - 10% gagnent plus)',
        'CENTILE_95': '95e centile (Top 5%)',
        'CENTILE_99': '99e centile (Top 1%)',
        '_T': 'Ensemble',
    },
}

## Fonction de nettoyage

On vérifie qu'il n'y a pas d'espaces en trop, on renomme les colonnes et vérifie que l’année et la valeur sont bien numériques.

In [ ]:
def nettoyer_donnees(df, nom_jeu):
    donnees = df.copy()
    donnees.columns = donnees.columns.str.strip().str.upper()

    colonnes_requises = {'TIME_PERIOD', 'OBS_VALUE'}
    manquantes = colonnes_requises.difference(donnees.columns)
    if manquantes:
        raise ValueError(
            f'{nom_jeu} : colonnes obligatoires absentes : {sorted(manquantes)}'
        )

    for colonne in donnees.select_dtypes(include=['object']).columns:
        donnees[colonne] = donnees[colonne].str.strip().replace('', pd.NA)

    donnees = donnees.rename(columns=RENOMMAGE)
    invalides = {}
    for colonne in ('annee', 'valeur'):
        numerique = pd.to_numeric(donnees[colonne], errors='coerce')
        invalides[colonne] = int((donnees[colonne].notna() & numerique.isna()).sum())
        donnees[colonne] = numerique

    donnees['annee'] = donnees['annee'].astype('Int64')

    for colonne, recodage in MODALITES.items():
        if colonne in donnees.columns:
            donnees[colonne] = donnees[colonne].replace(recodage)

    bilan = {
        'jeu': nom_jeu,
        'lignes': len(donnees),
        'colonnes': len(donnees.columns),
        'valeurs_manquantes': int(donnees.isna().sum().sum()),
        'doublons_exacts': int(donnees.duplicated().sum()),
        'annees_invalides': invalides['annee'],
        'valeurs_invalides': invalides['valeur'],
    }
    return donnees, bilan



## Vérification & Visualisation



In [8]:
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

donnees_brutes = pd.read_csv(CHEMIN_PRIVE, sep=';', encoding='utf-8-sig')
donnees_prive, bilan_prive = nettoyer_donnees(donnees_brutes, 'prive')

CHEMIN_SORTIE = DOSSIER_SORTIE / f'{CHEMIN_PRIVE.stem}_nettoye.csv'
donnees_prive.to_csv(CHEMIN_SORTIE, sep=';', index=False, encoding='utf-8-sig')
print(f'Export créé -> {CHEMIN_SORTIE}')

bilan_qualite = pd.DataFrame([bilan_prive]).set_index('jeu')
display(bilan_qualite)


print('Aperçu — données privées')
display(donnees_prive.head())

print(f'Fichier nettoyé enregistré dans : {DOSSIER_SORTIE}')


/tmp/ipykernel_55206/332595520.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for colonne in donnees.select_dtypes(include=['object']).columns:


Export créé -> /workspaces/ProjetPython/Data nettoyees/DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv


,lignes,colonnes,valeurs_manquantes,doublons_exacts,annees_invalides,valeurs_invalides
jeu,,,,,,
prive,14768,14,0,0,0,0


Aperçu — données privées


,geo,zone_geo,frequence,secteur_activite,tranche_age,taille_entreprise,categorie_socio_prof,sexe,temps_travail,centile_salaire,indicateur,unite,annee,valeur
0,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Femme,Temps complet,Ensemble,Salaire net moyen mensuel (Euros constants),Euros constants,1961,816.00
1,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Homme,Temps complet,Ensemble,Salaire net moyen mensuel (Euros constants),Euros constants,1952,826.00
2,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Homme,Temps complet,Ensemble,Évolution du salaire net mensuel (%),Pourcentage,1973,4.75
3,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Homme,Temps complet,99e centile (Top 1%),Salaire net moyen mensuel (Euros constants),Euros constants,2010,11286.00
4,F,FRANCE,A,Ensemble des secteurs,Ensemble,Ensemble,Ensemble,Femme,Temps complet,Ensemble,Évolution du salaire net mensuel (%),Pourcentage,1963,5.60


Fichier nettoyé enregistré dans : /workspaces/ProjetPython/Data nettoyees
